# The Hill cipher

A polygraphic substitution cipher: plaintext is cut into blocks of `n` letters, each block
is read as a column vector `x`, and the ciphertext block is `K · x (mod 26)`.

Decryption needs the modular inverse of `K`, which exists only when `det(K)` is coprime
with 26. This notebook uses the library in `src/applied_crypto/hill_cipher.py`.


## 1. Build the cipher

The 2x2 key `[[3, 3], [2, 5]]` has determinant 9, which is coprime with 26, so the key
is invertible. The constructor validates that and raises `InvalidKeyError` otherwise.


In [ ]:
import sys
sys.path.insert(0, '../src')

from applied_crypto import HillCipher, InvalidKeyError

key = [[3, 3], [2, 5]]
cipher = HillCipher(key)
print('block size :', cipher.block_size)
print('determinant:', cipher.determinant)
print('invertible :', cipher.is_invertible)


## 2. Encrypt and decrypt

Input is lower-cased and stripped of everything outside `a-z`, and the final block is
padded so its length is a multiple of the block size.


In [ ]:
message = 'help'
secret = cipher.encrypt(message)
print('plaintext :', message)
print('ciphertext:', secret)
print('recovered :', cipher.decrypt(secret))


### A 3x3 key

The same API works for any block size. The three-letter blocks below come from the key
used in the original coursework.


In [ ]:
wide = HillCipher([[3, 10, 20], [20, 9, 17], [9, 4, 17]])
encrypted = wide.encrypt('kita lawan')
print('ciphertext:', encrypted)
print('recovered :', wide.decrypt(encrypted))


## 3. Why a key gets rejected

A key is unusable when `det(K)` shares a factor with 26 — that is, when it is even or a
multiple of 13. Roughly a third of all matrices fail this test, so it is worth checking
before choosing one.


In [ ]:
for candidate in ([[3, 3], [2, 5]], [[1, 2], [3, 4]], [[13, 0], [0, 1]]):
    try:
        accepted = HillCipher(candidate)
        print(f'{candidate} -> usable, det={accepted.determinant}')
    except InvalidKeyError as error:
        print(f'{candidate} -> rejected: {error}')


## 4. Encrypting a corpus

`data/sample_messages.txt` holds a few lines to run through the cipher.


In [ ]:
from pathlib import Path

messages = Path('../data/sample_messages.txt').read_text().strip().splitlines()
for line in messages:
    secret = wide.encrypt(line)
    assert wide.decrypt(secret).startswith(line.replace(' ', ''))
    print(f'{line:20} -> {secret}')


## Where this is weak

The Hill cipher is linear: a known-plaintext attack recovers the key from `n`
plaintext/ciphertext block pairs by solving a linear system mod 26. It also leaks the
plaintext's letter frequencies modulo the block size. It is a teaching cipher, not a
secure one — `tests/test_hill_cipher.py` covers correctness, not security.
